In [24]:
from openrouteservice import client
from shapely.geometry import Point
from diskcache import Cache
import folium
import hashlib
import json

In [ ]:
API_KEY = "YOUR_ORS_API_KEY"
ors_client = client.Client(key=API_KEY)

start_coords = [79.8447, 6.9271]  # Galle Face Green
end_coords   = [79.8612, 6.9130]  # National Museum of Colombo

# We convert these to (Lon, Lat)
roadblocks_raw = [
    (6.92336, 79.84816),
    (6.92302, 79.85245)
]


In [26]:
def get_relevant_blockers(start, end, all_blockers, buffer_degrees=0.05):
    """
    Only keeps blockers within the bounding box of your trip
    plus a small buffer (approx 5km).
    """
    min_lon = min(start[0], end[0]) - buffer_degrees
    max_lon = max(start[0], end[0]) + buffer_degrees
    min_lat = min(start[1], end[1]) - buffer_degrees
    max_lat = max(start[1], end[1]) + buffer_degrees

    return [
        (lat, lon) for lat, lon in all_blockers
        if min_lat <= lat <= max_lat and min_lon <= lon <= max_lon
    ]

In [27]:
def normalize_coords(obj, precision=5):
    """
    Recursively rounds all floating point numbers in a
    GeoJSON-like structure to ensure cache consistency.
    """
    if isinstance(obj, float):
        return round(obj, precision)
    if isinstance(obj, dict):
        return {k: normalize_coords(v, precision) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [normalize_coords(x, precision) for x in obj]
    return obj

In [ ]:
cache = Cache('cache')

def get_smart_route(start, end, all_roadblocks):
    # Filter only relevant blockers to increase cache hits
    relevant_blockers = get_relevant_blockers(start, end, all_roadblocks)

    # Sort and Normalize to ensure the key is stable
    relevant_blockers.sort()
    key_data = {
        "start": normalize_coords(start),
        "end": normalize_coords(end),
        "blocks": normalize_coords(relevant_blockers)
    }

    # Hash and Check Cache
    key = hashlib.md5(json.dumps(key_data, sort_keys=True).encode()).hexdigest()

    if key in cache:
        return cache[key]

    current_avoid_polygons = {
        "type": "MultiPolygon",
        "coordinates": [
            [list(Point(lon, lat).buffer(0.0003).exterior.coords)]
            for lat, lon in relevant_blockers
        ]
    }

    route_params = {
        'coordinates': [
            normalize_coords(start_coords),
            normalize_coords(end_coords)
        ],
        'profile': 'driving-car',
        'format': 'geojson',
        'options': {
            'avoid_polygons': normalize_coords(current_avoid_polygons)
        }
    }

    # If Miss, Call API
    # (Convert relevant blockers to avoid_polygons here...)
    route = ors_client.directions(**route_params)
    cache[key] = route
    return route

In [29]:

route = get_smart_route(start_coords, end_coords, roadblocks_raw)

# Visualization
m = folium.Map(location=[6.9200, 79.8550], zoom_start=15, max_zoom=15, min_zoom=15, tiles="cartodbpositron")

# Add Start/End Markers
folium.Marker([6.9271, 79.8447], popup="Galle Face", icon=folium.Icon(color='green')).add_to(m)
folium.Marker([6.9130, 79.8612], popup="Museum", icon=folium.Icon(color='red')).add_to(m)

# Add Roadblocks
for lat, lon in roadblocks_raw:
    folium.Circle(location=[lat, lon], radius=20, color='red', fill=True, popup="Blocker").add_to(m)

folium.GeoJson(route, name="Colombo Route").add_to(m)

m